# Publish a Model: My Titanic Survival Classifier

My custom Phase 5 project, based on the M6 serving example.

- Author: Venkat Teja Nallamothu
- Date: 2026-08
- Dataset: Seaborn Titanic
- Target: survived

Run all cells top to bottom (**Run All**) before pushing to GitHub.

## Overview

The example project uses the penguins dataset to predict `species`.

My project uses the **Titanic** dataset to predict `survived`:
did a given passenger survive the sinking (1) or not (0)?

- supervised ML problem (I chose a target: `survived`)
- a **binary classification** problem (the target has exactly two categories)

Features: `pclass`, `sex`, `age`, `sibsp`, `parch`, `fare`.
`sex` arrives as the strings `"male"`/`"female"` and is encoded to `0`/`1`
before training - the same encoding is enforced again at serving time.

## Section 1. Project Setup and Imports

In [1]:
# === Section 1a. DECLARE IMPORTS ===

from typing import Any  # for type hinting

import joblib
import pandas as pd

# NOTE: mlstudio.serve_teja loads the saved model artifact as soon as it is
# imported, so it can't be imported here - the artifact doesn't exist yet.
# It is imported in Section 5, after the model has been trained and saved.
# === Section 1b. USE THE MODULE'S LOGGER (configured on import above) ===
from mlstudio.model_builder_teja import (
    DATASET_NAME,
    LOG,
    MODEL_PATH,
    TARGET_COL,
    load_data,
    save_model,
    split_data,
    summarize,
    train_model,
)

# === Section 1c. SET PANDAS DISPLAY CONFIGURATION (helps in notebooks) ===

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# === Section 1d. GLOBAL CONSTANTS AND CONFIGURATION ===

LOG.info(f"Model artifact will be saved to: {MODEL_PATH}")

2026-08-11 00:24:13 | INFO | M06 | === RUN START ===


2026-08-11 00:24:13 | INFO | M06 | project=M06


2026-08-11 00:24:13 | INFO | M06 | repo_dir=ml-06-serving


2026-08-11 00:24:13 | INFO | M06 | python=3.14.0


2026-08-11 00:24:13 | INFO | M06 | os=Windows 11


2026-08-11 00:24:13 | INFO | M06 | shell=powershell


2026-08-11 00:24:13 | INFO | M06 | cwd=notebooks


2026-08-11 00:24:13 | INFO | M06 | github_actions=False


2026-08-11 00:24:13 | INFO | M06 | Model artifact will be saved to: artifacts\model_teja.joblib


## Section 2. Load and Prepare the Data

In [2]:
# === Section 2. Load the Data ===

LOG.info(f"Loading dataset: {DATASET_NAME}")
df_model: pd.DataFrame = load_data()
LOG.info(f"Model rows: {df_model.shape[0]}")
LOG.info(f"Classes in '{TARGET_COL}': {sorted(df_model[TARGET_COL].unique())}")

2026-08-11 00:24:13 | INFO | M06 | Loading dataset: titanic


2026-08-11 00:24:13 | INFO | M06 | Loading dataset: titanic


2026-08-11 00:24:13 | INFO | M06 | Loaded: 891 rows, 15 columns


2026-08-11 00:24:13 | INFO | M06 | Model rows (after dropping missing): 714


2026-08-11 00:24:13 | INFO | M06 | Model rows: 714


2026-08-11 00:24:13 | INFO | M06 | Classes in 'survived': [np.int64(0), np.int64(1)]


## Section 3. Split into Train and Test

In [3]:
# === Section 3. Split into Train and Test ===

X_train, X_test, y_train, y_test = split_data(df_model)
LOG.info(f"Train instances: {len(X_train)}")
LOG.info(f"Test instances:  {len(X_test)}")

2026-08-11 00:24:13 | INFO | M06 | Train instances: 571


2026-08-11 00:24:13 | INFO | M06 | Test instances:  143


2026-08-11 00:24:13 | INFO | M06 | Train instances: 571


2026-08-11 00:24:13 | INFO | M06 | Test instances:  143


## Section 4. Train, Save, Reload Model

In [4]:
# === Section 4. Train, Save, and Reload ===

model = train_model(X_train, y_train)
save_model(model)
model = joblib.load(MODEL_PATH)
LOG.info(f"Reloaded model from: {MODEL_PATH}")

2026-08-11 00:24:13 | INFO | M06 | Training RandomForestClassifier on 571 instances


2026-08-11 00:24:14 | INFO | M06 | Training complete


2026-08-11 00:24:14 | INFO | M06 | Saved model to: artifacts\model_teja.joblib


2026-08-11 00:24:14 | INFO | M06 | Reloaded model from: artifacts\model_teja.joblib


## Section 5. Test the Serving Core

In [5]:
# === Section 5. Test the Serving Core ===

# Imported here (not in Section 1) because serve_teja loads the model
# artifact at import time, and that artifact was just saved in Section 4.
from mlstudio.serve_teja import predict_from_features

# valid payload - should return a prediction
good_payload: dict[str, Any] = {
    "pclass": 1,
    "sex": "female",
    "age": 29.0,
    "sibsp": 0,
    "parch": 0,
    "fare": 100.0,
}
result: dict[str, Any] = predict_from_features(model, good_payload)
LOG.info(f"Valid payload -> {result}")

# invalid payload - missing a required feature - should raise a clean ValueError
bad_payload: dict[str, Any] = {"pclass": 1, "sex": "female"}
try:
    predict_from_features(model, bad_payload)
    LOG.warning("Expected a ValueError for the bad payload but none was raised.")
except ValueError as exc:
    LOG.info(f"Invalid payload handled cleanly -> ValueError: {exc}")

# invalid payload - unrecognized 'sex' value - should also raise a clean ValueError
bad_sex_payload: dict[str, Any] = {
    "pclass": 1,
    "sex": "unknown",
    "age": 29.0,
    "sibsp": 0,
    "parch": 0,
    "fare": 100.0,
}
try:
    predict_from_features(model, bad_sex_payload)
    LOG.warning("Expected a ValueError for the bad sex value but none was raised.")
except ValueError as exc:
    LOG.info(f"Invalid sex value handled cleanly -> ValueError: {exc}")

2026-08-11 00:24:15 | INFO | M06 | === RUN START ===


2026-08-11 00:24:15 | INFO | M06 | project=M06


2026-08-11 00:24:15 | INFO | M06 | repo_dir=ml-06-serving


2026-08-11 00:24:15 | INFO | M06 | python=3.14.0


2026-08-11 00:24:15 | INFO | M06 | os=Windows 11


...

2026-08-11 00:24:15 | INFO | M06 | Loading model from: artifacts\model_teja.joblib


2026-08-11 00:24:15 | INFO | M06 | Model loaded successfully


C:\GitHub_Repo\ml-06-serving\.venv\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
2026-08-11 00:24:15 | INFO | M06 | Valid payload -> {'prediction': 1, 'survived': True}


2026-08-11 00:24:15 | INFO | M06 | Invalid payload handled cleanly -> ValueError: Missing required feature: 'age'


2026-08-11 00:24:15 | INFO | M06 | Invalid sex value handled cleanly -> ValueError: Invalid feature value for 'sex': expected one of ['male', 'female'], got 'unknown'


## Section 6. Summary and Next Steps

First, output key information (may use Python)
Second, provide your narrative, conclusions, and next steps (in Markdown)

In [6]:
# === Section 6. Summary ===

# Python summary
summarize()

2026-08-11 00:24:15 | INFO | M06 | ========================


2026-08-11 00:24:15 | INFO | M06 | SUMMARY


2026-08-11 00:24:15 | INFO | M06 | ========================


2026-08-11 00:24:15 | INFO | M06 | Dataset:  titanic


2026-08-11 00:24:15 | INFO | M06 | Target:   survived


2026-08-11 00:24:15 | INFO | M06 | Features: ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare']


2026-08-11 00:24:15 | INFO | M06 | Artifact: artifacts\model_teja.joblib


2026-08-11 00:24:15 | INFO | M06 | ========================


## Section 7. Visualize Feature Importance

Which passenger attributes drove the model's survival predictions?

In [7]:
# === Section 7. Feature Importance Chart ===

from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # save to file only - no blocking chart windows in the notebook

import matplotlib.pyplot as plt
import seaborn as sns

from mlstudio.model_builder_teja import FEATURE_COLS

CHART_PATH = Path("..") / "docs" / "images" / "teja_feature_importance.png"

importance_df = pd.DataFrame(
    {"feature": FEATURE_COLS, "importance": model.feature_importances_}
).sort_values("importance", ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=importance_df, x="importance", y="feature", ax=ax)
ax.set_title("Titanic Survival Model - Feature Importance")
ax.set_xlabel("Importance")
ax.set_ylabel("Feature")

CHART_PATH.parent.mkdir(parents=True, exist_ok=True)
fig.savefig(CHART_PATH)
LOG.info(f"Saved chart to: {CHART_PATH}")

plt.show()

2026-08-11 00:24:15 | INFO | M06 | Saved chart to: ..\docs\images\teja_feature_importance.png


C:\Users\royal\AppData\Local\Temp\ipykernel_9904\3958273058.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


### Custom Narrative

**My input contract:** a JSON object with all six features:
`pclass` (int), `sex` (`"male"` or `"female"`), `age` (float), `sibsp` (int),
`parch` (int), `fare` (float). A missing key or a `sex` value outside
`{"male", "female"}` raises a clean `ValueError` (→ HTTP 400 from the API)
instead of crashing the server.

**My response shape:** `{"prediction": 0 or 1, "survived": false or true}` -
the raw label plus a human-readable boolean, no probabilities exposed.

**Confirming the served model matches the saved one:** Section 4 saves the
model to `artifacts/model_teja.joblib` and immediately reloads it from that
same path with `joblib.load()` before Section 5 uses it to predict - so the
object being tested is provably the one that was written to disk, not just
the in-memory model from training.

### Next Steps

- Try `model.predict_proba()` to see how confident the model is, and decide
  whether the API should expose that probability alongside the label.
- Run the real FastAPI server (`uv run fastapi dev src/mlstudio/serve_teja.py`)
  and POST a few payloads with `curl` to see the actual HTTP status codes.